# Hurricane Melissa All-Land-Cover NDVI Greening Analysis

This notebook replaces the forest-equivalent greening denominator with all mapped 2013 land-cover area that had valid paired HLS NDVI and showed substantial post-storm greening. The main threshold is a relative NDVI increase >10% from the two-month pre-storm baseline, with pre-storm NDVI >= 0.20.


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7})


In [ ]:
BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "all_landcover_ndvi_greening"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NDVI_BEFORE_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
NDVI_AFTER_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
LANDCOVER_PATH = COMMON / "common_incoming_data" / "landcover" / "2013_landcover" / "2013_landuse_LandCover.shp"
RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"

J2USD = 1.0 / 150.0
REL_BASELINE_MIN = 0.20
REL_GREENING_THRESHOLD = 0.10
FIGURE_DPI = 300

for input_path in [NDVI_BEFORE_PATH, NDVI_AFTER_PATH, LANDCOVER_PATH, RIVER_EAD_MIN_PATH, RIVER_EAD_MAX_PATH]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR


In [ ]:
def clean_ndvi_array(ndvi_array: np.ndarray, nodata_value: float | None) -> np.ndarray:
    cleaned_array = ndvi_array.astype("float32")
    if nodata_value is not None and np.isfinite(nodata_value):
        cleaned_array[cleaned_array == nodata_value] = np.nan
    cleaned_array[~np.isfinite(cleaned_array)] = np.nan
    return cleaned_array


def landcover_group_for_class(class_name: str) -> str:
    if "Open dry forest" in class_name:
        return "Open dry forest"
    if class_name == "Plantation: Tree crops, shrub crops, sugar cane, banana" or "Hardwood Plantation" in class_name:
        return "Plantation / tree crops"
    if class_name in {
        "Secondary Forest",
        "Disturbed broadleaved forest (Secondary Forest)",
        "Closed broadleaved forest (Primary Forest)",
    }:
        return "Secondary / broadleaved forest"
    if class_name in {
        "Fields and Secondary Forest",
        "Bamboo and Secondary Forest",
        "Bamboo and Fields",
        "Fields  and Bamboo",
        "Fields or Secondary Forest/Pine Plantation",
    }:
        return "Mixed fields, bamboo and secondary forest"
    if class_name.startswith("Fields:"):
        return "Open / agricultural fields"
    if class_name in {"Bauxite Extraction", "Quarry"}:
        return "Bauxite extraction / quarry"
    if class_name == "Bamboo":
        return "Bamboo"
    if class_name in {"Mangrove Forest", "Swamp Forest", "Herbaceous Wetland", "Water Body"}:
        return "Wetland / water"
    if class_name == "Buildings and other infrastructures":
        return "Buildings and other infrastructure"
    if class_name == "Bare Rock":
        return "Bare rock"
    return "Other"


def pct(numerator: float, denominator: float) -> float:
    return float(numerator / denominator * 100.0) if denominator else np.nan


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * J2USD

    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")

    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan
    return ead_array, profile


def reproject_continuous_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    destination = np.full((reference_profile["height"], reference_profile["width"]), np.nan, dtype="float32")
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def ead_sum_usd(ead_array: np.ndarray, mask: np.ndarray) -> float:
    return float(np.nansum(np.where(mask & np.isfinite(ead_array), ead_array, 0.0)))


In [ ]:
with rasterio.open(NDVI_BEFORE_PATH) as ndvi_before_raster, rasterio.open(NDVI_AFTER_PATH) as ndvi_after_raster:
    ndvi_before = clean_ndvi_array(ndvi_before_raster.read(1), ndvi_before_raster.nodata)
    ndvi_after = clean_ndvi_array(ndvi_after_raster.read(1), ndvi_after_raster.nodata)
    ndvi_transform = ndvi_before_raster.transform
    ndvi_crs = ndvi_before_raster.crs
    ndvi_shape = (ndvi_before_raster.height, ndvi_before_raster.width)
    ndvi_pixel_area_ha = abs(ndvi_transform.a * ndvi_transform.e) / 10_000.0

    same_ndvi_grid = (
        ndvi_after_raster.transform == ndvi_before_raster.transform
        and ndvi_after_raster.crs == ndvi_before_raster.crs
        and ndvi_after_raster.shape == ndvi_before_raster.shape
    )
if not same_ndvi_grid:
    raise ValueError("Before and after NDVI rasters are not aligned")

paired_ndvi_mask = np.isfinite(ndvi_before) & np.isfinite(ndvi_after)
ndvi_eligible_mask = paired_ndvi_mask & (ndvi_before >= REL_BASELINE_MIN)
relative_ndvi_change = np.full(ndvi_shape, np.nan, dtype="float32")
np.divide(
    ndvi_after - ndvi_before,
    ndvi_before,
    out=relative_ndvi_change,
    where=ndvi_eligible_mask,
)
substantial_greening_mask = ndvi_eligible_mask & (relative_ndvi_change > REL_GREENING_THRESHOLD)
any_positive_change_mask = paired_ndvi_mask & (ndvi_after > ndvi_before)

landcover = gpd.read_file(LANDCOVER_PATH, columns=["Classify", "geometry"]).to_crs(ndvi_crs)
landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
landcover["Classify"] = landcover["Classify"].astype(str)
landcover["class_id"] = pd.factorize(landcover["Classify"], sort=True)[0] + 1
landcover_lookup = landcover[["class_id", "Classify"]].drop_duplicates().sort_values("class_id")

landcover_class_grid = rasterize(
    ((geometry, int(class_id)) for geometry, class_id in zip(landcover.geometry, landcover["class_id"])),
    out_shape=ndvi_shape,
    transform=ndvi_transform,
    fill=0,
    dtype="int16",
)
landcover_mask = landcover_class_grid > 0
substantial_greening_landcover_mask = substantial_greening_mask & landcover_mask
any_positive_change_landcover_mask = any_positive_change_mask & landcover_mask

ndvi_mask_summary = pd.DataFrame(
    [
        {
            "metric": "paired_ndvi_on_2013_landcover",
            "area_ha": float((paired_ndvi_mask & landcover_mask).sum() * ndvi_pixel_area_ha),
        },
        {
            "metric": "ndvi_eligible_baseline_ge_0p20_on_2013_landcover",
            "area_ha": float((ndvi_eligible_mask & landcover_mask).sum() * ndvi_pixel_area_ha),
        },
        {
            "metric": "relative_ndvi_increase_gt10pct_on_2013_landcover",
            "area_ha": float(substantial_greening_landcover_mask.sum() * ndvi_pixel_area_ha),
        },
        {
            "metric": "any_positive_ndvi_increase_on_2013_landcover",
            "area_ha": float(any_positive_change_landcover_mask.sum() * ndvi_pixel_area_ha),
        },
    ]
)
ndvi_mask_summary


In [ ]:
def summarise_mask_by_landcover(mask: np.ndarray, area_column: str) -> pd.DataFrame:
    maximum_class_id = int(landcover_lookup["class_id"].max())
    class_pixel_counts = np.bincount(landcover_class_grid[mask].ravel(), minlength=maximum_class_id + 1)
    total_area_ha = float(class_pixel_counts.sum() * ndvi_pixel_area_ha)
    rows = []
    for landcover_record in landcover_lookup.itertuples(index=False):
        class_area_ha = float(class_pixel_counts[int(landcover_record.class_id)] * ndvi_pixel_area_ha)
        if class_area_ha <= 0:
            continue
        rows.append(
            {
                "Classify": landcover_record.Classify,
                "landcover_group": landcover_group_for_class(landcover_record.Classify),
                area_column: class_area_ha,
                "pct_of_total_area": pct(class_area_ha, total_area_ha),
            }
        )
    return pd.DataFrame(rows).sort_values(area_column, ascending=False).reset_index(drop=True)


substantial_greening_by_class = summarise_mask_by_landcover(substantial_greening_landcover_mask, "area_ha")
substantial_greening_by_group = (
    substantial_greening_by_class.groupby("landcover_group", as_index=False)["area_ha"].sum()
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
substantial_greening_area_ha = substantial_greening_by_group["area_ha"].sum()
substantial_greening_by_group["pct_of_total_area"] = substantial_greening_by_group["area_ha"].map(
    lambda area_ha: pct(area_ha, substantial_greening_area_ha)
)

any_positive_change_by_class = summarise_mask_by_landcover(any_positive_change_landcover_mask, "area_ha")
any_positive_change_by_group = (
    any_positive_change_by_class.groupby("landcover_group", as_index=False)["area_ha"].sum()
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
any_positive_change_area_ha = any_positive_change_by_group["area_ha"].sum()
any_positive_change_by_group["pct_of_total_area"] = any_positive_change_by_group["area_ha"].map(
    lambda area_ha: pct(area_ha, any_positive_change_area_ha)
)

print("Relative NDVI increase >10% by grouped 2013 land cover")
display(substantial_greening_by_group.round(3))
print("Top detailed 2013 land-cover classes")
display(substantial_greening_by_class.head(20).round(3))


In [ ]:
NON_ECOLOGICAL_GREENING_CLASSES = {
    "Buildings and other infrastructures",
    "Bare Rock",
    "Water Body",
    "Bauxite Extraction",
    "Quarry",
}

excluded_greening_by_class = substantial_greening_by_class[
    substantial_greening_by_class["Classify"].isin(NON_ECOLOGICAL_GREENING_CLASSES)
].copy()
ecologically_interpretable_greening_by_class = substantial_greening_by_class[
    ~substantial_greening_by_class["Classify"].isin(NON_ECOLOGICAL_GREENING_CLASSES)
].copy()

excluded_greening_area_ha = excluded_greening_by_class["area_ha"].sum()
ecologically_interpretable_greening_area_ha = ecologically_interpretable_greening_by_class["area_ha"].sum()

ecology_percentage_denominator = ecologically_interpretable_greening_area_ha
ecologically_interpretable_greening_by_class["pct_of_ecologically_interpretable_greening_area"] = (
    ecologically_interpretable_greening_by_class["area_ha"] / ecology_percentage_denominator * 100.0
)
excluded_greening_by_class["pct_of_all_gt10_greening_area"] = (
    excluded_greening_by_class["area_ha"] / substantial_greening_area_ha * 100.0
)

ecologically_interpretable_greening_by_group = (
    ecologically_interpretable_greening_by_class.groupby("landcover_group", as_index=False)["area_ha"].sum()
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
ecologically_interpretable_greening_by_group["pct_of_ecologically_interpretable_greening_area"] = (
    ecologically_interpretable_greening_by_group["area_ha"] / ecology_percentage_denominator * 100.0
)

excluded_greening_summary = pd.DataFrame(
    [
        {
            "metric": "all_gt10_ndvi_greening_area",
            "area_ha": substantial_greening_area_ha,
            "pct_of_all_gt10_greening_area": 100.0,
        },
        {
            "metric": "excluded_non_ecological_or_extractive_classes",
            "area_ha": excluded_greening_area_ha,
            "pct_of_all_gt10_greening_area": pct(excluded_greening_area_ha, substantial_greening_area_ha),
        },
        {
            "metric": "ecologically_interpretable_gt10_greening_area",
            "area_ha": ecologically_interpretable_greening_area_ha,
            "pct_of_all_gt10_greening_area": pct(
                ecologically_interpretable_greening_area_ha,
                substantial_greening_area_ha,
            ),
        },
    ]
)

print("Non-ecological / extractive / water classes excluded from ecological interpretation")
display(excluded_greening_by_class[["Classify", "area_ha", "pct_of_all_gt10_greening_area"]].round(3))
print("Ecologically interpretable >10% greening by grouped 2013 land cover")
display(ecologically_interpretable_greening_by_group.round(3))


In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd = read_positive_ead_usd(RIVER_EAD_MAX_PATH, river_profile)[0]
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)

river_benefit_on_ndvi_grid = np.zeros(ndvi_shape, dtype="uint8")
reproject(
    source=river_benefit_mask.astype("uint8"),
    destination=river_benefit_on_ndvi_grid,
    src_transform=river_profile["transform"],
    src_crs=river_profile["crs"],
    src_nodata=0,
    dst_transform=ndvi_transform,
    dst_crs=ndvi_crs,
    dst_nodata=0,
    resampling=Resampling.nearest,
)

substantial_greening_river_benefit_overlap_ndvi_grid = substantial_greening_landcover_mask & (river_benefit_on_ndvi_grid == 1)
substantial_greening_river_benefit_overlap_ndvi_grid_area_ha = float(
    substantial_greening_river_benefit_overlap_ndvi_grid.sum() * ndvi_pixel_area_ha
)

river_ndvi_before = reproject_continuous_to_reference(NDVI_BEFORE_PATH, river_profile)
river_ndvi_after = reproject_continuous_to_reference(NDVI_AFTER_PATH, river_profile)
river_ndvi_eligible_mask = (
    river_benefit_mask
    & np.isfinite(river_ndvi_before)
    & np.isfinite(river_ndvi_after)
    & (river_ndvi_before >= REL_BASELINE_MIN)
)
river_relative_ndvi_change = np.full((river_profile["height"], river_profile["width"]), np.nan, dtype="float32")
np.divide(
    river_ndvi_after - river_ndvi_before,
    river_ndvi_before,
    out=river_relative_ndvi_change,
    where=river_ndvi_eligible_mask,
)
river_benefit_substantial_greening_mask = river_ndvi_eligible_mask & (river_relative_ndvi_change > REL_GREENING_THRESHOLD)
river_pixel_area_ha = abs(river_profile["transform"].a * river_profile["transform"].e) / 10_000.0
river_benefit_substantial_greening_area_ha = float(river_benefit_substantial_greening_mask.sum() * river_pixel_area_ha)

river_overlap_summary = pd.DataFrame(
    [
        {
            "metric": "all_landcover_gt10_greening_area",
            "area_ha": substantial_greening_area_ha,
            "pct_of_all_landcover_gt10_greening": 100.0,
            "avoided_ead_usd_minimum": np.nan,
            "avoided_ead_usd_maximum": np.nan,
            "note": "All mapped 2013 land-cover pixels with paired NDVI, baseline NDVI >= 0.20, and relative NDVI increase >10%.",
        },
        {
            "metric": "overlap_with_river_benefit_pixels_on_ndvi_grid",
            "area_ha": substantial_greening_river_benefit_overlap_ndvi_grid_area_ha,
            "pct_of_all_landcover_gt10_greening": pct(
                substantial_greening_river_benefit_overlap_ndvi_grid_area_ha,
                substantial_greening_area_ha,
            ),
            "avoided_ead_usd_minimum": np.nan,
            "avoided_ead_usd_maximum": np.nan,
            "note": "River-flood positive avoided-EAD mask reprojected to the NDVI grid with nearest-neighbour resampling.",
        },
        {
            "metric": "river_benefit_pixels_gt10_greening_on_native_river_grid",
            "area_ha": river_benefit_substantial_greening_area_ha,
            "pct_of_all_landcover_gt10_greening": pct(
                river_benefit_substantial_greening_area_ha,
                substantial_greening_area_ha,
            ),
            "avoided_ead_usd_minimum": ead_sum_usd(river_ead_min_usd, river_benefit_substantial_greening_mask),
            "avoided_ead_usd_maximum": ead_sum_usd(river_ead_max_usd, river_benefit_substantial_greening_mask),
            "note": "Native avoided-EAD grid calculation; use this when reporting avoided EAD values.",
        },
    ]
)
river_overlap_summary


In [ ]:
group_colors = {
    "Open / agricultural fields": "#bdbdbd",
    "Buildings and other infrastructure": "#636a70",
    "Open dry forest": "#9b6a2f",
    "Plantation / tree crops": "#2b8c57",
    "Mixed fields, bamboo and secondary forest": "#d8a934",
    "Secondary / broadleaved forest": "#66a61e",
    "Wetland / water": "#6baed6",
    "Bare rock": "#969696",
    "Bauxite extraction / quarry": "#c66b35",
    "Bamboo": "#7fcdbb",
}

plot_data = substantial_greening_by_group.sort_values("area_ha", ascending=True)
fig, axis = plt.subplots(figsize=(180 / 25.4, 95 / 25.4), dpi=FIGURE_DPI)
axis.barh(
    plot_data["landcover_group"],
    plot_data["area_ha"],
    color=plot_data["landcover_group"].map(group_colors),
    edgecolor="white",
    linewidth=0.35,
)
for row_position, plot_record in enumerate(plot_data.itertuples(index=False)):
    axis.text(
        plot_record.area_ha,
        row_position,
        f" {plot_record.pct_of_total_area:.1f}%",
        va="center",
        ha="left",
        fontsize=6,
    )
axis.set_xlabel("Area with relative NDVI increase >10% (ha)")
axis.set_title("Post-Hurricane Melissa NDVI greening by 2013 land cover", pad=4)
axis.grid(axis="x", linewidth=0.3, alpha=0.35)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.set_xlim(0, plot_data["area_ha"].max() * 1.18)
fig.tight_layout()

figure_paths = []
for suffix in ["png", "pdf", "svg"]:
    figure_path = OUT_DIR / f"hurricane_melissa_all_landcover_ndvi_greening_by_landcover.{suffix}"
    fig.savefig(figure_path, bbox_inches="tight", facecolor="white")
    figure_paths.append(figure_path)

display(fig)
plt.close(fig)
figure_paths


In [ ]:
output_tables = {
    "all_landcover_ndvi_mask_summary.csv": ndvi_mask_summary,
    "all_landcover_gt10_greening_by_class.csv": substantial_greening_by_class,
    "all_landcover_gt10_greening_by_group.csv": substantial_greening_by_group,
    "all_landcover_any_positive_ndvi_change_by_class.csv": any_positive_change_by_class,
    "all_landcover_any_positive_ndvi_change_by_group.csv": any_positive_change_by_group,
    "all_landcover_gt10_greening_river_benefit_overlap_summary.csv": river_overlap_summary,
    "ecologically_interpretable_gt10_greening_by_class.csv": ecologically_interpretable_greening_by_class,
    "ecologically_interpretable_gt10_greening_by_group.csv": ecologically_interpretable_greening_by_group,
    "excluded_gt10_greening_non_ecological_classes.csv": excluded_greening_by_class,
    "excluded_gt10_greening_summary.csv": excluded_greening_summary,
}
output_paths = []
for filename, output_table in output_tables.items():
    output_path = OUT_DIR / filename
    output_table.to_csv(output_path, index=False)
    output_paths.append(output_path)

output_paths


In [ ]:
def all_group_area(group_name: str) -> float:
    return float(substantial_greening_by_group.loc[substantial_greening_by_group["landcover_group"].eq(group_name), "area_ha"].sum())


def ecological_group_area(group_name: str) -> float:
    return float(
        ecologically_interpretable_greening_by_group.loc[
            ecologically_interpretable_greening_by_group["landcover_group"].eq(group_name),
            "area_ha",
        ].sum()
    )


open_agricultural_area_ha = ecological_group_area("Open / agricultural fields")
open_dry_forest_area_ha = ecological_group_area("Open dry forest")
plantation_area_ha = ecological_group_area("Plantation / tree crops")
mixed_fields_area_ha = ecological_group_area("Mixed fields, bamboo and secondary forest")
secondary_broadleaved_area_ha = ecological_group_area("Secondary / broadleaved forest")
wetland_water_area_ha = ecological_group_area("Wetland / water")
bamboo_area_ha = ecological_group_area("Bamboo")

native_river_overlap_record = river_overlap_summary.loc[
    river_overlap_summary["metric"].eq("river_benefit_pixels_gt10_greening_on_native_river_grid")
].iloc[0]
ndvi_grid_overlap_record = river_overlap_summary.loc[
    river_overlap_summary["metric"].eq("overlap_with_river_benefit_pixels_on_ndvi_grid")
].iloc[0]

summary_text = f"""Across all mapped 2013 land-cover classes, {substantial_greening_area_ha:,.1f} ha showed a relative NDVI increase >10% after Hurricane Melissa, using the same baseline condition of pre-storm NDVI >= {REL_BASELINE_MIN:.2f}. However, {excluded_greening_area_ha:,.1f} ha ({pct(excluded_greening_area_ha, substantial_greening_area_ha):.1f}%) occurred in classes that should not be interpreted as ecological greening: buildings and other infrastructure, bare rock, water body, bauxite extraction and quarry. These pixels are retained as a QA category because they may reflect mixed 30 m pixels, land-cover mismatch, or spectral artefacts rather than vegetation recovery.

Restricting interpretation to vegetated or revegetatable land-cover classes, {ecologically_interpretable_greening_area_ha:,.1f} ha showed a relative NDVI increase >10%. The largest share occurred in open/agricultural fields ({open_agricultural_area_ha:,.1f} ha; {pct(open_agricultural_area_ha, ecologically_interpretable_greening_area_ha):.1f}%), followed by open dry forest ({open_dry_forest_area_ha:,.1f} ha; {pct(open_dry_forest_area_ha, ecologically_interpretable_greening_area_ha):.1f}%) and plantation/tree crops ({plantation_area_ha:,.1f} ha; {pct(plantation_area_ha, ecologically_interpretable_greening_area_ha):.1f}%). Mixed fields, bamboo and secondary forest accounted for {mixed_fields_area_ha:,.1f} ha ({pct(mixed_fields_area_ha, ecologically_interpretable_greening_area_ha):.1f}%), secondary/broadleaved forest for {secondary_broadleaved_area_ha:,.1f} ha ({pct(secondary_broadleaved_area_ha, ecologically_interpretable_greening_area_ha):.1f}%), wetland vegetation classes for {wetland_water_area_ha:,.1f} ha ({pct(wetland_water_area_ha, ecologically_interpretable_greening_area_ha):.1f}%), and bamboo for {bamboo_area_ha:,.1f} ha ({pct(bamboo_area_ha, ecologically_interpretable_greening_area_ha):.1f}%).

Of the all-land-cover greening area, {native_river_overlap_record.area_ha:,.1f} ha, or {native_river_overlap_record.pct_of_all_landcover_gt10_greening:.1f}%, occurred within river-flood restoration-benefit pixels when assessed on the native avoided-EAD grid. This is equivalent to {pct(native_river_overlap_record.area_ha, ecologically_interpretable_greening_area_ha):.1f}% of the ecologically interpretable greening area. These greening benefit pixels accounted for US${native_river_overlap_record.avoided_ead_usd_minimum / 1e6:.2f}-{native_river_overlap_record.avoided_ead_usd_maximum / 1e6:.2f} million in avoided EAD. As a same-grid spatial overlay check, reprojecting the river-flood benefit mask to the NDVI grid gave {ndvi_grid_overlap_record.area_ha:,.1f} ha, or {ndvi_grid_overlap_record.pct_of_all_landcover_gt10_greening:.1f}% of the all-land-cover greening area.
"""

summary_path = OUT_DIR / "hurricane_melissa_all_landcover_ndvi_greening_interpretation.md"
summary_path.write_text(summary_text, encoding="utf-8")
print(summary_text)
print(f"Wrote: {summary_path}")
